# FoML viva notebook — Colab

Upload this file: **Colab → File → Upload notebook**.
Then **File → Save a copy in Drive** so a disconnect does not wipe it.

What this shows:
1. Linear regression two ways — closed-form (NumPy / sklearn) and **gradient descent (PyTorch)**
2. Decision tree with **entropy** (sklearn — not PyTorch; trees are not differentiable)
3. Short Git + Kaggle notes at the bottom for viva

Run with **Shift+Enter**. Runtime → Run all if a TA wants a full demo.


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

print("Python", sys.version.split()[0])
print("numpy", np.__version__, "torch", torch.__version__)


## 1. Linear regression — what it is

Model: $y = a_0 + a_1 x$. Fit by **least squares**: minimise $\sum_i (y_i - \hat y_i)^2$.

- Closed form: design matrix $X = [\mathbf{1},\, x]$, solve $X^\top X w = X^\top y$ (NumPy `lstsq`, sklearn `LinearRegression`).
- Iterative: same $w$, but walk downhill on MSE with **gradient descent** (PyTorch).

sklearn: `coef_` = $a_1$, `intercept_` = $a_0$. Metrics: MSE (lower better), $R^2$ (1 = perfect).


In [ ]:
x = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
y = np.array([1.5, 3.2, 4.1, 5.8, 6.9, 8.4, 9.1, 11.0], dtype=float)

# closed form: design matrix [1, x]
X_np = np.column_stack([np.ones_like(x), x])
a0, a1 = np.linalg.lstsq(X_np, y, rcond=None)[0]
print(f"numpy   y = {a0:.3f} + {a1:.3f} x")

X = x.reshape(-1, 1)
lin = LinearRegression().fit(X, y)
print(f"sklearn y = {lin.intercept_:.3f} + {lin.coef_[0]:.3f} x")
print("MSE", mean_squared_error(y, lin.predict(X)), "R^2", r2_score(y, lin.predict(X)))


## 2. Same line, PyTorch (gradient descent)

`nn.Linear(1, 1)` is $\hat y = wx + b$. Loop: predict → MSE → `zero_grad` → `backward` → `step`.

Viva: *"PyTorch is not a different model. It is a different solver. η too large → loss explodes."*


In [ ]:
torch.manual_seed(0)
xt = torch.tensor(x, dtype=torch.float32).reshape(-1, 1)
yt = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
opt = torch.optim.SGD(model.parameters(), lr=0.02)

history = []
for epoch in range(2000):
    y_hat = model(xt)
    loss = loss_fn(y_hat, yt)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 200 == 0:
        history.append((epoch, loss.item()))

b = model.bias.item()
w = model.weight.item()
print(f"pytorch y = {b:.3f} + {w:.3f} x   (final MSE {loss.item():.4f})")
print("loss every 200 epochs:", history)

plt.figure(figsize=(5, 3.5))
plt.scatter(x, y, label="data")
xs = np.linspace(x.min(), x.max(), 50)
plt.plot(xs, lin.predict(xs.reshape(-1, 1)), label="sklearn / lstsq")
plt.plot(xs, b + w * xs, "--", label="pytorch SGD")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.title("Linear regression")
plt.show()


## 3. Decision tree — entropy (sklearn, not PyTorch)

Kurhekar play/go-out table. Split by **information gain**: $H = -\sum p \log_2 p$, gain = $H_{parent} - $ weighted $H_{children}$.

Root should be **Weather**. Cloudy → Yes (pure). Sunny → Humidity. Rainy → Wind.

Trees are **not** trained in PyTorch: a split is a discrete `if`, so there is no gradient. Use `DecisionTreeClassifier(criterion="entropy")`.


In [ ]:
rows = [
    ["Sunny",  "Hot",  "High",   "Weak",   "No"],
    ["Cloudy", "Hot",  "High",   "Weak",   "Yes"],
    ["Sunny",  "Mild", "Normal", "Strong", "Yes"],
    ["Cloudy", "Mild", "High",   "Strong", "Yes"],
    ["Rainy",  "Mild", "High",   "Strong", "No"],
    ["Rainy",  "Cool", "Normal", "Strong", "No"],
    ["Rainy",  "Mild", "High",   "Weak",   "Yes"],
    ["Sunny",  "Hot",  "High",   "Strong", "No"],
    ["Cloudy", "Hot",  "Normal", "Weak",   "Yes"],
    ["Rainy",  "Mild", "High",   "Strong", "No"],
]
cols = ["weather", "temp", "humidity", "wind"]
X_cat = np.array([r[:4] for r in rows])
y_cat = np.array([r[4] for r in rows])

encoders = {c: LabelEncoder().fit(X_cat[:, i]) for i, c in enumerate(cols)}
X_num = np.column_stack([encoders[c].transform(X_cat[:, i]) for i, c in enumerate(cols)])
y_enc = LabelEncoder().fit(y_cat)
y_num = y_enc.transform(y_cat)

tree = DecisionTreeClassifier(criterion="entropy", random_state=0)
tree.fit(X_num, y_num)
pred = tree.predict(X_num)
print("train accuracy (n=10)", accuracy_score(y_num, pred))
print("confusion\\n", confusion_matrix(y_num, pred, labels=y_enc.transform(["No", "Yes"])))
print("Do not trust 100% on n=10 — overfitting. Need a held-out test set.")

plt.figure(figsize=(10, 6))
plot_tree(tree, feature_names=cols, class_names=list(y_enc.classes_), filled=True)
plt.title("Decision tree (entropy)")
plt.show()


## 4. Git (say this, do not over-explain)

| command | meaning |
|---|---|
| `git clone <url>` | copy a GitHub repo onto this machine |
| `git status` | what changed |
| `git add file` | stage for the next snapshot |
| `git commit -m "msg"` | save a **local** snapshot |
| `git push` | upload commits to GitHub |
| `git pull` | download others' commits |

Git = the tool. GitHub = the website. A commit is a snapshot with a message. `.gitignore` keeps data / secrets out of the repo.

Colab: **File → Save a copy in GitHub**, or Download `.ipynb` and `git add` it locally.


## 5. Kaggle (open these before you walk in)

- Site: [kaggle.com](https://www.kaggle.com/) — datasets, competitions, hosted notebooks (like Colab).
- **Titanic** ([competition](https://www.kaggle.com/competitions/titanic)): classification — did the passenger survive? Sex, class, age. Logistic regression or a tree. Metric: accuracy.
- **House Prices** ([competition](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)): regression — predict sale price. Linear regression is the baseline; trees usually beat it. Metric: RMSE / RMSLE.

Workflow: `train.csv` → clean / encode → **train/val split** → `fit` → predict `test.csv` → submit. Never train on the test labels.
